# GPT-2 S SFT full-user PLUM pipeline

Main SFT run for a GPT-2 Small/Base model initialized from the weakest available CPT branch: `cpt_gpt2_v1/checkpoint-1400`.

This notebook runs the canonical PLUM-style path:

1. Load Semantic IDs and CPT tokenizer/checkpoint.
2. Build supervised next-item SID examples.
3. Fine-tune GPT-2 S for 3 epochs.
4. Decode top-K SID candidates with trie constraints.
5. Map generated SIDs back to MovieLens item IDs.
6. Report Recall/NDCG/MRR plus invalid SID and coverage diagnostics.

Protocol note: this is the full-user leave-one-out SFT run: one train target per user, plus full validation and test targets for all users. Full sliding-window SFT would create about 970k train examples and should be treated as a separate larger experiment.

## Imports and configuration

In [ ]:
import json
import sys
import time
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import GPT2LMHeadModel, GPT2TokenizerFast, set_seed

from src.sft import (
    SFTDataPaths,
    SFTDatasetBuilder,
    SFTExampleConfig,
    SFTSchema,
    SFTTrainingConfig,
    SIDMapping,
    evaluate_rankings,
    examples_to_dataset,
    generate_recommendations,
    train_sft,
)

set_seed(42)
torch.backends.cuda.matmul.allow_tf32 = True

ROOT

In [ ]:
RUN_NAME = "sft_gpt2_s_weak_cpt_3epochs"

SID_PATH = ROOT / "data" / "processed" / "SIDs_V1.npy"
CPT_CHECKPOINT = ROOT / "data" / "processed" / "artifacts" / "CPT_user_behavior_V1" / "cpt_gpt2_v1" / "checkpoint-1400"
OUTPUT_DIR = ROOT / "data" / "processed" / "artifacts" / "SFT" / RUN_NAME
REPORT_PATH = ROOT / "reports" / f"{RUN_NAME}_metrics.json"

MAX_HISTORY_EVENTS = 100
MAX_LENGTH = 512
NUM_EPOCHS = 3
NUM_BEAMS = 20
TOP_K = 20

{
    "sid_path": str(SID_PATH),
    "cpt_checkpoint": str(CPT_CHECKPOINT),
    "output_dir": str(OUTPUT_DIR),
    "report_path": str(REPORT_PATH),
}

## Build full-user SFT datasets

In [ ]:
sids = np.load(SID_PATH)
tokenizer = GPT2TokenizerFast.from_pretrained(CPT_CHECKPOINT)

schema = SFTSchema(n_sid_levels=sids.shape[1])
builder = SFTDatasetBuilder(
    paths=SFTDataPaths(
        processed_dir=ROOT / "data" / "processed",
        raw_ml1m_dir=ROOT / "data" / "raw" / "ml-1m",
    ),
    schema=schema,
    config=SFTExampleConfig(
        max_history_events=MAX_HISTORY_EVENTS,
        min_history_events=3,
        max_length=MAX_LENGTH,
        include_user_features=True,
        include_ratings=True,
        train_examples_per_user=1,
        max_users=None,
    ),
)

train, val, test, users = builder.load_tables()
train_examples = builder.build_train_examples(sids=sids, tokenizer=tokenizer, train=train, users=users)
val_examples = builder.build_eval_examples("val", sids=sids, tokenizer=tokenizer, train=train, val=val, test=test, users=users)
test_examples = builder.build_eval_examples("test", sids=sids, tokenizer=tokenizer, train=train, val=val, test=test, users=users)

len(train_examples), len(val_examples), len(test_examples)

## Invariant checks

In [ ]:
def assert_sft_example(example):
    assert example["target_item_idx"] not in example["history_item_idx"]
    assert len(example["target_sid"]) == sids.shape[1]
    assert len(example["input_ids"]) == len(example["labels"])
    assert any(label != -100 for label in example["labels"])
    first_target = next(i for i, label in enumerate(example["labels"]) if label != -100)
    assert first_target == example["prompt_length"]
    assert all(label == -100 for label in example["labels"][:first_target])

for example in train_examples[:256] + val_examples[:256] + test_examples[:256]:
    assert_sft_example(example)

sid_mapping = SIDMapping.from_sids(sids, interactions=train)
{
    "sid_uniqueness": sid_mapping.uniqueness,
    "collision_buckets": sid_mapping.n_collision_buckets,
    "collided_items_in_collision_buckets": sid_mapping.n_collided_items,
    "collision_excess": sid_mapping.n_collision_excess,
    "max_train_len": max(len(x["input_ids"]) for x in train_examples),
    "max_val_len": max(len(x["input_ids"]) for x in val_examples),
    "max_test_len": max(len(x["input_ids"]) for x in test_examples),
}

## Train GPT-2 S from weakest CPT for 3 epochs

In [ ]:
train_ds = examples_to_dataset(train_examples)
val_ds = examples_to_dataset(val_examples)

# Keep teacher-forcing eval loss affordable during training. Full ranking eval comes after training.
val_loss_ds = val_ds.select(range(min(512, len(val_ds))))

training_config = SFTTrainingConfig(
    model_name_or_path=CPT_CHECKPOINT,
    tokenizer_name_or_path=CPT_CHECKPOINT,
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    max_steps=-1,
    learning_rate=1e-5,
    warmup_steps=50,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    fp16=False,
    bf16=torch.cuda.is_available(),
    gradient_checkpointing=False,
    save_total_limit=2,
    disable_tqdm=True,
)

started_at = time.time()
result = train_sft(train_ds, val_loss_ds, training_config)
train_seconds = time.time() - started_at
result["train_output"].metrics | {"train_seconds": train_seconds}

## Full constrained decoding evaluation

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_tokenizer = GPT2TokenizerFast.from_pretrained(OUTPUT_DIR)
eval_model = GPT2LMHeadModel.from_pretrained(OUTPUT_DIR).to(device)
eval_model.eval()

def evaluate_split(split_name, examples, max_examples=None):
    selected = examples[:max_examples] if max_examples is not None else examples
    records = []
    for example in tqdm(selected, desc=f"decode {split_name}"):
        records.append(
            generate_recommendations(
                model=eval_model,
                tokenizer=eval_tokenizer,
                example=example,
                sid_mapping=sid_mapping,
                schema=schema,
                sids=sids,
                k=TOP_K,
                num_beams=NUM_BEAMS,
                constraint="trie",
                device=device,
            )
        )
    metrics = evaluate_rankings(records, k_values=(1, 5, 10, 20))
    return records, metrics

val_records, val_metrics = evaluate_split("val", val_examples)
test_records, test_metrics = evaluate_split("test", test_examples)

val_metrics, test_metrics

## Save metrics and predictions

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

def compact_record(record):
    return {
        "user_id": record["user_id"],
        "target_item_idx": record["target_item_idx"],
        "target_sid": list(record["target_sid"]),
        "candidates": record["candidates"],
        "invalid_sid_count": record["invalid_sid_count"],
        "generated_sid_count": record["generated_sid_count"],
        "duplicate_count": record["duplicate_count"],
    }

def write_jsonl(path, records):
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(compact_record(record), ensure_ascii=False) + "\n")

write_jsonl(OUTPUT_DIR / "val_predictions.jsonl", val_records)
write_jsonl(OUTPUT_DIR / "test_predictions.jsonl", test_records)

summary = {
    "run_name": RUN_NAME,
    "protocol": "full-user leave-one-out SFT; one train target per user",
    "model_family": "GPT-2 S/base",
    "cpt_checkpoint": str(CPT_CHECKPOINT),
    "output_dir": str(OUTPUT_DIR),
    "sid_path": str(SID_PATH),
    "sid_shape": list(sids.shape),
    "train_examples": len(train_examples),
    "val_examples": len(val_examples),
    "test_examples": len(test_examples),
    "num_epochs": NUM_EPOCHS,
    "num_beams": NUM_BEAMS,
    "top_k": TOP_K,
    "train_metrics": result["train_output"].metrics,
    "train_seconds": train_seconds,
    "val_metrics": val_metrics,
    "test_metrics": test_metrics,
}

REPORT_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
summary